## Pipeline de simulación de repertorios inmunológicos

Este pipeline permite generar repertorios inmunológicos simulados 
con parámetros controlados, facilitando el análisis bioinformático 
de diversidad clonotípica bajo distintos escenarios.

El flujo de trabajo integra herramientas en **R y Python**, 
coordinadas mediante un script en **Bash (.sh)**, lo que asegura 
automatización, reproducibilidad y trazabilidad en todas las etapas 
del procesamiento.

Los análisis en R se ejecutan y documentan mediante un notebook 
interactivo (`.ipynb`), mientras que el script Bash gestiona la 
ejecución de módulos en Python y la orquestación general del pipeline.

### Procesamiento de secuencias

Los repertorios simulados generados deben ser procesados mediante 
el framework **Immcantation**, el cual permite realizar la 
**asignación clonal y el clustering de secuencias**, generando 
los clonotipos utilizados en los análisis posteriores.

### Salida del pipeline

El resultado final del pipeline es un archivo de clonotipos 
(`clone-pass.tsv`), que contiene la asignación de cada secuencia 
a un clon, y que sirve como entrada para el cálculo de métricas 
de diversidad.

## Configuración del entorno reproducible con renv

Se utilizó el paquete **renv** para gestionar un entorno 
reproducible del proyecto en R, permitiendo aislar y fijar 
las versiones exactas de los paquetes utilizados.

Este enfoque asegura que los análisis puedan ser replicados 
en otros sistemas sin problemas de compatibilidad.

### Inicialización del entorno

- `renv::init()` → inicializa el proyecto y genera el archivo `renv.lock`, 
  que almacena las versiones de los paquetes utilizados.

### Instalación de paquetes

Se instalaron paquetes desde:

- **CRAN**: `dplyr`, `ggplot2`, `viridisLite`, `readr`, `here`
- **Bioconductor**: `Biostrings`, `IRanges`, `GenomicRanges`

Estos paquetes fueron instalados usando `install.packages()` 
y `BiocManager::install()` según corresponda.

### Control de versiones

- `renv::snapshot()` → guarda las versiones exactas de los paquetes 
  en el archivo `renv.lock`.

- `renv::restore()` → permite restaurar el entorno en otro computador 
  con las mismas versiones de paquetes.

Este flujo garantiza la **reproducibilidad completa del análisis**.

In [2]:
# pak::pkg_install("BiocManager")
# BiocManager::install("Biostrings")
# pak::pkg_install("immuneSIM")
library(Biostrings)
library(immuneSIM)
library(here)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, is.unsorted, lapply,
    mapply, match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    rank, rbind, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: 'S4Vectors'


The following object is masked from 'package:utils':

    findMatches


The follo

## Simulación de repertorios con immuneSIM

Se generaron repertorios simulados de secuencias de inmunoglobulinas 
utilizando el paquete **immuneSIM**, configurando distintos parámetros 
que controlan la composición, diversidad y características biológicas 
de las secuencias.

### Descripción de variables

- **name_repertoire**: nombre del repertorio simulado  
- **number_of_seqs**: número total de secuencias generadas  
- **species**: especie (ej. humano)  
- **receptor**: tipo de receptor (`ig` para BCR, `tr` para TCR)  
- **chain**: tipo de cadena (pesada o liviana)  
- **verbose**: activación de mensajes durante la simulación  

### Parámetros de distribución clonal

- **equal_cc**: define si todos los clonotipos tienen el mismo tamaño  
- **user_defined_alpha**: controla la uniformidad de la distribución clonal  
  (valores altos generan mayor desigualdad entre clones)

### Mutaciones somáticas (SHM)

- **shm**: modelo de mutación somática. Puede tomar los valores:

  - `none`: no se simulan mutaciones  
  - `poisson`: mutaciones aleatorias sin sesgo  
  - `data`: basado en perfiles reales (más mutaciones en regiones CDR)  
  - `naive`: secuencias sin SHM (puede incluir artefactos técnicos)  
  - `motif`: mutaciones dirigidas por motivos específicos  

- **shm.prob**: probabilidad de mutación por secuencia  
  (ej. 15/350 ≈ 15 mutaciones en 350 nucleótidos)

### Parámetros de recombinación V(D)J

- **vdj_noise**: introduce variabilidad en la selección de genes V, D y J  
  (rango 0–1; valores más altos → mayor aleatoriedad)

### Longitud de CDR3

- **max_cdr3_length / min_cdr3_length**: límites para la longitud de CDR3  

En humanos, la longitud típica de CDR3 en cadenas pesadas (IgH) 
varía entre 10 y 25 aminoácidos, con una mediana aproximada 
de 15–16.

In [ ]:

# SIMULACIÓN ESCENARIO 1
# Cargar librerías

# Parámetro: número de secuencias

n_seqs <- 102400  # Cambiar a 10, 100, 1000, etc.

# Simular repertorio A CONTROL

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
  number_of_seqs  = n_seqs,
  species         = "hs",
  receptor        = "ig",
  chain           = "h",
  verbose         = TRUE,
  equal_cc        = TRUE,
  shm             = "none",
  shm.prob        = 0,
  vdj_noise       = 0,
  max_cdr3_length = 15,
  min_cdr3_length = 15,

)


# Preparar secuencias para FASTA

dna_sequences <- Biostrings::DNAStringSet(sim_repertoire$sequence)
names(dna_sequences) <- paste0("seq", seq_along(dna_sequences))


# Guardar archivo FASTA en 'data/input'

out_file <- here("..", "data", "input",
                 paste0("repertorio_A_insilico_", n_seqs, "_seqs.fasta"))

writeXStringSet(dna_sequences, filepath = out_file)
message("Archivo guardado en: ", out_file)



initializing sim..
simulated sequences: 10000 
simulated sequences: 20000 
simulated sequences: 30000 
simulated sequences: 40000 
simulated sequences: 50000 
simulated sequences: 60000 
simulated sequences: 70000 
simulated sequences: 80000 
simulated sequences: 90000 
simulated sequences: 1e+05 


Archivo guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks/../data/input/repertorio_A_insilico_102400_seqs.fasta



In [ ]:
# SIMULACIÓN ESCENARIO 2
# Cargar librerías


# Parámetro: número de secuencias

n_seqs <- 102400 # Cambiar a 10, 100, 1000, etc.

# Simular repertorio  B naive 

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 3,  
  shm = "none",
  shm.prob = 0,
  vdj_noise = 0,
  max_cdr3_length =20, 
  min_cdr3_length =6,
  
)


# Preparar secuencias para FASTA

dna_sequences <- Biostrings::DNAStringSet(sim_repertoire$sequence)
names(dna_sequences) <- paste0("seq", seq_along(dna_sequences))


# Guardar archivo FASTA en 'data/input'

out_file <- here("..", "data", "input",
                 paste0("repertorio_B_insilico_", n_seqs, "_seqs.fasta"))

writeXStringSet(dna_sequences, filepath = out_file)
message("Archivo guardado en: ", out_file)




initializing sim..
simulated sequences: 10000 
simulated sequences: 20000 
simulated sequences: 30000 
simulated sequences: 40000 
simulated sequences: 50000 
simulated sequences: 60000 
simulated sequences: 70000 
simulated sequences: 80000 
simulated sequences: 90000 
simulated sequences: 1e+05 


Archivo guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks/../data/input/repertorio_B_insilico_102400_seqs.fasta



In [ ]:
# SIMULACIÓN ESCENARIO 3
# Cargar librerías


# Parámetro: número de secuencias

n_seqs <- 102400  # Cambiar a 10, 100, 1000, etc.

# Simular repertorio C mixto sangre periferica con expansión y  SHM en R

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 2,  
  shm = "data",
  shm.prob = 20/350,
  vdj_noise = 0,
  max_cdr3_length =18, 
  min_cdr3_length =10,
  
)

# Preparar secuencias para FASTA

dna_sequences <- Biostrings::DNAStringSet(sim_repertoire$sequence)
names(dna_sequences) <- paste0("seq", seq_along(dna_sequences))



# Guardar archivo FASTA en 'data/input'

out_file <- here("..", "data", "input",
                 paste0("repertorio_C_insilico_", n_seqs, "_seqs.fasta"))

writeXStringSet(dna_sequences, filepath = out_file)
message("Archivo guardado en: ", out_file)





initializing sim..
simulated sequences: 10000 
simulated sequences: 20000 
simulated sequences: 30000 
simulated sequences: 40000 
simulated sequences: 50000 
simulated sequences: 60000 
simulated sequences: 70000 
simulated sequences: 80000 
simulated sequences: 90000 
simulated sequences: 1e+05 


Archivo guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks/../data/input/repertorio_C_insilico_102400_seqs.fasta



In [ ]:
# SIMULACIÓN ESCENARIO 4
# Cargar librerías


# Parámetro: número de secuencias

n_seqs <- 102400  # Cambiar a 10, 100, 1000, etc.

# Simular repertorio D folicular (centro germinal)  alta SHM,
# selección por afinidad, expansión marcada de pocos clones, CDR3 con rango amplio en R

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 1.5,  
  shm = "motif",
  shm.prob = 40/350,
  vdj_noise = 0,
  max_cdr3_length =15, 
  min_cdr3_length =13,
  )

# Preparar secuencias para FASTA

dna_sequences <- Biostrings::DNAStringSet(sim_repertoire$sequence)
names(dna_sequences) <- paste0("seq", seq_along(dna_sequences))



# Guardar archivo FASTA en 'data/input'

out_file <- here("..", "data", "input",
                 paste0("repertorio_D_insilico_", n_seqs, "_seqs.fasta"))

writeXStringSet(dna_sequences, filepath = out_file)
message("Archivo guardado en: ", out_file)


initializing sim..
simulated sequences: 10000 
simulated sequences: 20000 
simulated sequences: 30000 
simulated sequences: 40000 
simulated sequences: 50000 
simulated sequences: 60000 
simulated sequences: 70000 
simulated sequences: 80000 
simulated sequences: 90000 
simulated sequences: 1e+05 


Archivo guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks/../data/input/repertorio_D_insilico_102400_seqs.fasta



In [ ]:
# SIMULACIÓN ESCENARIO 5
# Cargar librerías


# Parámetro: número de secuencias

n_seqs <- 102400  # Cambiar a 10, 100, 1000, etc.

# Simular repertorio E extrafolicular (fuera del centro germinal) expansión moderada, 
# shm moderada y baja.

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 2.5,  
  shm = "poisson",
  shm.prob = 30/350,
  vdj_noise = 0,
  max_cdr3_length =17, 
  min_cdr3_length =13,
 
  )
  # Preparar secuencias para FASTA 

dna_sequences <- Biostrings::DNAStringSet(sim_repertoire$sequence)
names(dna_sequences) <- paste0("seq", seq_along(dna_sequences))


# Guardar archivo FASTA en 'data/input'

out_file <- here("..", "data", "input",
                 paste0("repertorio_E_insilico_", n_seqs, "_seqs.fasta"))

writeXStringSet(dna_sequences, filepath = out_file)
message("Archivo guardado en: ", out_file)


initializing sim..
simulated sequences: 10000 
simulated sequences: 20000 
simulated sequences: 30000 
simulated sequences: 40000 
simulated sequences: 50000 
simulated sequences: 60000 
simulated sequences: 70000 
simulated sequences: 80000 
simulated sequences: 90000 
simulated sequences: 1e+05 


Archivo guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks/../data/input/repertorio_E_insilico_102400_seqs.fasta



## Clustering y asignación clonal

Se ejecutó el script `testscript.sh` para realizar el 
clustering de secuencias y la asignación clonal a partir 
del archivo FASTA de entrada, generando los clonotipos 
utilizados en los análisis posteriores.

In [ ]:
# CODIGO DE LLAMADO 
INPUT_FILE="repertorio_E_insilico_102400_seqs.fasta"
# Ejecuta el pipeline
!cd ../scripts/ && bash testscript.sh 

 Iniciando pipeline Immcantation...
 Archivo de entrada detectado: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/scripts/../data/input/repertorio_E_insilico_102400_seqs.fasta
 Ejecutando AssignGenes.py...
   START> AssignGenes
 COMMAND> igblast
 VERSION> 1.22.0
    FILE> repertorio_E_insilico_102400_seqs.fasta
ORGANISM> human
    LOCI> ig
   NPROC> 4

PROGRESS> 09:43:56 |Done                     | 22.8 min

  PASS> 102400
OUTPUT> repertorio_E_insilico_102400_seqs_igblast.fmt7
   END> AssignGenes

 AssignGenes.py completado: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/scripts/../data/output/repertorio_E_insilico_102400_seqs_igblast.fmt7
 Ejecutando MakeDb.py...
         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> repertorio_E_insilico_102400_seqs_igblast.fmt7
      SEQ_FILE> repertorio_E_insilico_102400_seqs.fasta
         NPROC> 4
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 09:43:58